# Composing multiple plots

plotlet's composition primitives:

  * `a | b` — side-by-side
  * `a / b` — stacked
  * `pt.grid([[a, b], [c, d]])` — true 2-D grids (and irregular ones with `None`)

These are *operators on Charts* — composition is component-first: each leaf carries a `data_width=` / `data_height=` and the parent figure size is the sum (plus gaps). Once a chart is in a parent you render the parent, never the child.


In [1]:
import plotlet as pt
import math

xs = [i * 0.1 for i in range(64)]
sin = [math.sin(t) for t in xs]
cos = [math.cos(t) for t in xs]


## Side by side: `a | b`


In [2]:
a = pt.chart(title='sin', data_width=290, data_height=300)
a.line(xs, sin)
b = pt.chart(title='cos', data_width=290, data_height=300)
b.line(xs, cos)
a | b


## Stacked: `a / b`

Same idea, vertical. Choose dimensions so the totals land where you want them.


In [3]:
top = pt.chart(title='sin', data_width=600, data_height=140)
top.line(xs, sin)
bot = pt.chart(title='cos', data_width=600, data_height=140)
bot.line(xs, cos)
top / bot


## Mixing: `(a | b) / c`

Parens control nesting. `a | b | c` flattens into a single 3-cell row (left-fold), but cross-direction composition stays nested.


In [4]:
a = pt.chart(title='sin', data_width=290, data_height=140); a.line(xs, sin)
b = pt.chart(title='cos', data_width=290, data_height=140); b.line(xs, cos)
c = pt.chart(title='sum', data_width=600, data_height=140)
c.line(xs, [s + co for s, co in zip(sin, cos)])
(a | b) / c


## `pt.grid` for 2-D layouts

`pt.grid` is the right tool when `|` / `/` get awkward. Columns and rows align across the grid: column width is the max canvas across cells in that column, row height likewise. Use `None` for empty cells. To make a column 2× wider than another, set `data_width=` per leaf — body-first sizing replaces the old `widths=` / `heights=` ratio overrides.


In [5]:
def panel(title, ys, w=180, h=180):
    c = pt.chart(title=title, data_width=w, data_height=h)
    c.line(xs, ys)
    return c

pt.grid([
    [panel('a', sin),                         panel('b', cos)],
    [panel('c', [s + 1 for s in sin]),        panel('d', [c - 1 for c in cos])],
])


## Sharing scales: `.share_x()` / `.share_y()`

When `.share_x()` is called on a parent (or `share_x=True/'col'/'row'` is passed to `pt.grid`), plotlet:

  1. **adopts** `other`'s x-domain (sharer's data does not extend the shared scale)
  2. **collapses the gap** between the joined pair (no inner spine, ticks, or labels — but the *outer* spines and ticks stay)
  3. **propagates join state** through grid columns / rows, so a shared boundary stays aligned with un-shared neighbors

Stacked time series — share `x`:


In [6]:
top = pt.chart(title='sin', data_height=140); top.line(xs, sin)
bot = pt.chart(title='cos', data_height=200); bot.line(xs, cos)
(top / bot).share_x()


Side by side comparison — share `y`:


In [7]:
left  = pt.chart(title='small',  data_width=300, data_height=240)
left.line(xs, [s * 0.4 for s in sin])
right = pt.chart(title='full',   data_width=300, data_height=240)
right.line(xs, sin)
(left | right).share_y()


## Sizing in composition

Each leaf's `width` / `height` is treated as a relative ratio when composing. A narrow side panel sharing y with a wide main panel is just `pt.chart(data_width=80)` composed with `.share_y()`:


In [8]:
import random
random.seed(0)
ny, nx = 24, 40
z = [[math.sin(i * 0.3) * math.cos(j * 0.2) + random.gauss(0, 0.1)
      for j in range(nx)] for i in range(ny)]
row_means = [sum(row) / len(row) for row in z]

main = pt.chart(title='heatmap', data_width=520, data_height=300, ylabel='row')
main.imshow(z, cmap='RdBu_r')
side = pt.chart(title='row mean', data_width=80, data_height=300)
side.line(row_means, list(range(ny)))
(main | side).share_y()


## Annotated heatmap: top track + main + left dendrogram-stand-in

The point of share_x / share_y plus grid join propagation: irregular layouts where one row's column boundary is joined (because of a share) but another row's matching boundary isn't. The grid keeps everything column-aligned anyway.


In [9]:
col_means = [sum(z[i][j] for i in range(ny)) / ny for j in range(nx)]
row_means = [sum(row) / len(row) for row in z]

main = pt.chart(data_width=440, data_height=260)
main.imshow(z, cmap='RdBu_r')

top  = pt.chart(title='col mean', data_width=440, data_height=80)
top.line(list(range(nx)), col_means)

left = pt.chart(title='row mean', data_width=120, data_height=260)
left.line(row_means, list(range(ny)))

pt.grid([
    [None, top ],
    [left, main],
])


## Render once, render the parent

After composition, the leaves are *owned* by their parent. Calling `.show()` / `.to_svg()` on a child raises — the child has no Figure of its own; its SVG only makes sense in the context of the parent layout.


In [10]:
a = pt.chart(); a.line([1,2,3], [1,2,3])
b = pt.chart(); b.line([1,2,3], [3,2,1])
parent = a | b
try:
    a.to_svg()
except RuntimeError as e:
    print('child raises:', e)
print('parent renders fine:', len(parent.to_svg()), 'chars of SVG')


child raises: this chart is part of a composed parent; render the parent instead.
parent renders fine: 88892 chars of SVG
